# REMAP-CAP design modules: competency questions (identity only)

Each competency question is a SPARQL query in `queries/`, run over `output/inferred.ttl` (the modules
merged and classified with ELK, `scripts/05_reason.py`). Each answer is checked against pandas computed
from the step 3 identity tables. A mismatch stops the notebook.

Scope: Core 4.0 and 4.1, PANDA 1, and the nine PANDA-era DSAs (`scripts/rcap_docs.py`): Corticosteroid (CS),
Antibiotic (AB), Tropical Antibiotic (TAB), Influenza Antiviral (IAV), Macrolide Duration (MD), COVID-19
Immunoglobulin (IG), Mechanical Ventilation (MV), Endothelial Modulation (END), Influenza Immune Modulation (IIM).
Identity is exact text (whitespace collapsed) with Kerstin's crosswalk decisions applied
(`sources/crosswalk_decisions.csv`). The last section compares
exact identity, the decisions, and an upper bound with every review pair merged.

R4 and R5 use the delegation and default readings (step 5 part 2). R2 uses the readings of both parts.


In [1]:
import re
import sys
from pathlib import Path

import pandas as pd
import rdflib

BASE = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
NS = "https://kerfors.github.io/early-clinical-design-research/remapcap-design-modules/"
sys.path.insert(0, str(BASE / "scripts"))
from rcap_docs import DSAS, SHORT  # scripts/rcap_docs.py
NOT_DSA = ["CORE40", "CORE41", "PANDA"]

graph = rdflib.Graph()
graph.parse(BASE / "output" / "inferred.ttl")
occ = pd.read_csv(BASE / "output" / "step3" / "occurrences.csv", dtype=str, keep_default_na=False)
rev = pd.read_csv(BASE / "output" / "step3" / "core_revisions.csv", dtype=str, keep_default_na=False)
sections = pd.read_csv(BASE / "output" / "step1" / "sections.csv", dtype=str, keep_default_na=False)
docs_of = occ.groupby("component").doc.apply(lambda v: frozenset(v))
text_of = occ.drop_duplicates("component").set_index("component").text


def short(value):
    return str(value).replace(NS, "rcap:")


def run(name):
    query = (BASE / "queries" / name).read_text()
    rows = [[short(x) if x is not None else "" for x in r] for r in graph.query(query)]
    return rows


# the DSA lists in the queries (VALUES, R1 count) equal DSAS
dsa_iris = {"rcap:DomainDesign_" + SHORT[d] for d in DSAS}
for name in ["r1_common_to_all_dsas.rq", "r2e_per_dsa.rq", "r3_dsa_pairs.rq"]:
    text = (BASE / "queries" / name).read_text()
    blocks = re.findall(r"VALUES \?(?:d|dsa|a|b) \{([^}]*)\}", text)
    assert blocks and all(set(b.split()) == dsa_iris for b in blocks), name
assert re.search(r"COUNT\(DISTINCT \?d\) = (\d+)", (BASE / "queries" / "r1_common_to_all_dsas.rq").read_text()).group(1) == str(len(DSAS))
print(len(graph), "triples;", len(DSAS), "DSAs")


30529 triples; 9 DSAs


## Classification

Expected from the build rules: both Core versions are inferred under the version-independent
`CoreDesign`; each DSA design is asserted under `CoreDesign` (unversioned binding) and is **not**
inferred under either Core version.


In [2]:
hier = pd.read_csv(BASE / "output" / "design_hierarchy.csv")
core_versions = hier[hier.design.isin(["rcap:CoreDesign_4_0", "rcap:CoreDesign_4_1"])]
assert set(core_versions.superclass) == {"rcap:CoreDesign"} and set(core_versions.status) == {"inferred"}
dsa_rows = hier[hier.design.str.startswith("rcap:DomainDesign_")]
assert not dsa_rows.superclass.isin(["rcap:CoreDesign_4_0", "rcap:CoreDesign_4_1"]).any()
assert set(dsa_rows[dsa_rows.superclass == "rcap:CoreDesign"].status) == {"asserted"}
hier


,design,superclass,status
0,rcap:CoreDesign_4_0,rcap:CoreDesign,inferred
1,rcap:CoreDesign_4_1,rcap:CoreDesign,inferred
2,rcap:DomainDesign_AB,rcap:CoreDesign,asserted
3,rcap:DomainDesign_AB,rcap:SharedDSA_AB_MD,inferred
4,rcap:DomainDesign_AB,rcap:SharedDSA_AB_TAB,inferred
5,rcap:DomainDesign_AB,rcap:SharedDSA_CS_AB_TAB_IAV_MD_IG_MV_END_IIM,inferred
6,rcap:DomainDesign_CS,rcap:CoreDesign,asserted
7,rcap:DomainDesign_CS,rcap:SharedDSA_CS_AB_TAB_IAV_MD_IG_MV_END_IIM,inferred
8,rcap:DomainDesign_CS,rcap:SharedDSA_CS_IAV,inferred
9,rcap:DomainDesign_CS,rcap:SharedDSA_CS_IIM,inferred


## R1: components in all DSAs in scope and in no Core version and no PANDA

The query works on the modules, whose Core components cover only the Core sections mapped to component
kinds. The second check compares each answer with the full text of every Core 4.0, Core 4.1 and PANDA
section, so a hit is not an artefact of the section mapping.


In [3]:
r1 = pd.DataFrame(run("r1_common_to_all_dsas.rq"), columns=["component", "text"])
expected = sorted("rcap:" + c for c, d in docs_of.items() if d == frozenset(DSAS))
assert sorted(r1.component) == expected, (sorted(r1.component), expected)

full = sections[sections.doc.isin(NOT_DSA)].text
wordings = occ.groupby("component").text.apply(set)
r1["in_full_core_or_panda_text"] = r1.component.map(
    lambda c: any(full.str.contains(re.escape(t)).any() for t in wordings[c[5:]]))
r1["n_wordings"] = r1.component.map(lambda c: len(wordings[c[5:]]))
assert not r1.in_full_core_or_panda_text.any()
r1["kinds"] = r1.component.map(lambda c: ";".join(sorted(set(occ[occ.component == c[5:]].kind))))
r1


,component,text,in_full_core_or_panda_text,n_wordings,kinds
0,rcap:c324186f8e2,The treating clinician believes that participa...,False,2,domain_exclusion
1,rcap:c85858c8378,Refer to relevant Core Protocol documents for ...,False,6,discontinuation
2,rcap:cc4a3982cda,Patients will be excluded from this domain if ...,False,1,domain_exclusion
3,rcap:cc4efc1a692,Interactions with all other domains are either...,False,3,allocation_interactions


## R3: DSA pairs sharing components beyond Core and PANDA

Shared = both DSAs have part of it, and no Core version and no PANDA has part of it.


In [4]:
r3 = pd.DataFrame(run("r3_dsa_pairs.rq"), columns=["a", "b", "shared"])
r3["shared"] = r3.shared.astype(int)
rows = []
for i, a in enumerate(DSAS):
    for b in DSAS[i + 1:]:
        n = sum(1 for d in docs_of if {a, b} <= d and not (d & set(NOT_DSA)))
        pair = sorted(["rcap:DomainDesign_" + SHORT[a], "rcap:DomainDesign_" + SHORT[b]])
        rows.append({"a": pair[0], "b": pair[1], "expected": n})
check = r3.merge(pd.DataFrame(rows), on=["a", "b"], how="outer")
assert (check.shared == check.expected).all(), check
check.sort_values("shared", ascending=False)


,a,b,shared,expected
7,rcap:DomainDesign_AB,rcap:DomainDesign_TAB,58,58
17,rcap:DomainDesign_END,rcap:DomainDesign_IIM,26,26
11,rcap:DomainDesign_CS,rcap:DomainDesign_IIM,24,24
22,rcap:DomainDesign_IAV,rcap:DomainDesign_IIM,24,24
4,rcap:DomainDesign_AB,rcap:DomainDesign_IIM,24,24
9,rcap:DomainDesign_CS,rcap:DomainDesign_IAV,24,24
1,rcap:DomainDesign_AB,rcap:DomainDesign_END,23,23
20,rcap:DomainDesign_END,rcap:DomainDesign_TAB,20,20
32,rcap:DomainDesign_IIM,rcap:DomainDesign_TAB,20,20
15,rcap:DomainDesign_END,rcap:DomainDesign_IAV,18,18


## R6 (module level): what a DSA inherits from a Core that changed

A DSA design is a subclass of the version-independent `CoreDesign`. For a Core component whose text
differs between 4.0 and 4.1, `CoreDesign` has part only the version-independent parent: every DSA
inherits the component, and neither version's content (R6a).
R6b lists Core components that differ between the versions without a text change (moved: same text,
other section and kind) and components in one version only; `CoreDesign` does not have part of those.


In [5]:
r6a = pd.DataFrame(run("r6a_inherited_revised.rq"), columns=["dsa", "parent", "text_40", "text_41"])
revised = rev[rev.status == "revised"]
assert len(r6a) == len(revised) * len(DSAS)
assert set(r6a.text_40) == set(revised.text_40) and set(r6a.text_41) == set(revised.text_41)
r6a.drop_duplicates("parent")[["parent", "text_40", "text_41"]]


,parent,text_40,text_41
0,rcap:r52076d4ff9,"For RAR, allocation proportions will be capped...","For RAR, allocation proportions will be capped..."
1,rcap:r87900e2424,*High-flow oxygen defined as a fractional insp...,*High-flow oxygen is defined as a fractional i...
2,rcap:rcdb4b81960,Collected while participants are admitted to h...,Collected while participants are admitted to h...


In [6]:
r6b = pd.DataFrame(run("r6b_core_role_or_version_only.rq"),
                   columns=["component", "status", "kinds_40", "kinds_41", "sections_40", "sections_41"])
exp = rev[rev.status.isin(["moved", "only_40", "only_41"])]
assert r6b.status.value_counts().to_dict() == exp.status.value_counts().to_dict()
moved = r6b[r6b.status == "moved"]
assert (moved.kinds_40 != moved.kinds_41).all()
print(r6b.status.value_counts().to_dict())
moved.assign(text=moved.component.map(lambda c: text_of[c[5:]][:90]))


{'only_41': 30, 'only_40': 14, 'moved': 8}


,component,status,kinds_40,kinds_41,sections_40,sections_41,text
0,rcap:c4ea2e1b3d0,moved,endpoint_primary,endpoint_exploratory,7.6.1,7.6.3,"Landmark mortality at Day 90 post-randomization,"
1,rcap:c876d85ebab,moved,endpoint_primary,endpoint_exploratory,7.6.1,7.6.3,This hierarchical ordinal scale is a composite...
2,rcap:c8d1e9454aa,moved,endpoint_primary,endpoint_exploratory,7.6.1,7.6.3,Hospitalized on non-invasive ventilation or hi...
3,rcap:c9f97c0e55a,moved,endpoint_primary,endpoint_exploratory,7.6.1,7.6.3,"Among Day-90 survivors, their daily respirator..."
4,rcap:cc66e2d09a4,moved,endpoint_primary,endpoint_exploratory,7.6.1,7.6.3,Hospitalized on low-intensity oxygen
5,rcap:ce3b8327b04,moved,endpoint_primary,endpoint_exploratory,7.6.1,7.6.3,"Hospitalized, no oxygen"
6,rcap:c44479053d2,moved,endpoint_secondary,endpoint_primary,7.6.2,7.6.1,"Participants who have been, and still are, adm..."
7,rcap:c9fd4c2c7f1,moved,endpoint_secondary,endpoint_primary,7.6.2,7.6.1,The index hospital admission is defined as con...


## R6 (statement level) and R7: readings

From `modules/readings.ttl` (`scripts/09_readings_module.py`): one `InheritanceReading` per DSA statement that
takes content from the Core Protocol or core protocol documents, resolved by `sources/topic_rules.csv`
(rules agreed 2026-10-24, title rules accepted 2026-10-25). V-current: an unversioned statement resolves to
Core 4.1. V-transition: Core 4.1 7.6.2 ("DSAs dated before this Core Protocol amendment will additionally
report the pre-specified endpoints ... previous versions of the Core Protocol"), so a DSA dated before Core
4.1 whose statement resolves to 7.6.1 or 7.6.2 resolves to 4.1 and, additionally, 4.0. All analyst readings.


In [7]:
r6s = pd.DataFrame(run("r6s_statements_on_changed_core.rq"), columns=["dsa", "section", "target", "change", "versions"])
sv = pd.read_csv(BASE / "output" / "step5" / "statement_versions.csv", dtype=str, keep_default_na=False)
assert len(r6s) == len(sv)
exp = sv.assign(dsa="rcap:DomainDesign_" + sv.doc.map(SHORT)).groupby("dsa").size()
assert r6s.groupby("dsa").size().to_dict() == exp.to_dict()
r6s.groupby("dsa").size().rename("statements on Core sections changed in 4.1")


dsa
rcap:DomainDesign_AB      4
rcap:DomainDesign_CS     10
rcap:DomainDesign_END     7
rcap:DomainDesign_IAV     8
rcap:DomainDesign_IG      5
rcap:DomainDesign_IIM     8
rcap:DomainDesign_MD      5
rcap:DomainDesign_MV      3
rcap:DomainDesign_TAB     4
Name: statements on Core sections changed in 4.1, dtype: int64

In [8]:
r7 = pd.DataFrame(run("r7_two_core_versions.rq"), columns=["dsa", "versions", "n"])
r7["n"] = r7.n.astype(int)
exp = sv.assign(dsa="rcap:DomainDesign_" + sv.doc.map(SHORT)).groupby(["dsa", "resolves_to_core_versions"]).size()
assert {(r.dsa, r.versions): r.n for r in r7.itertuples()} == exp.to_dict()
r7.pivot(index="dsa", columns="versions", values="n").fillna(0).astype(int)


versions,4.1,4.1 + 4.0 (additionally)
dsa,,
rcap:DomainDesign_AB,2,2
rcap:DomainDesign_CS,6,4
rcap:DomainDesign_END,4,3
rcap:DomainDesign_IAV,8,0
rcap:DomainDesign_IG,2,3
rcap:DomainDesign_IIM,5,3
rcap:DomainDesign_MD,2,3
rcap:DomainDesign_MV,1,2
rcap:DomainDesign_TAB,2,2


## R4 and R5: delegations and defaults

From `modules/delegations.ttl` (`scripts/11_delegation_module.py`): one `DelegationReading` per Core 4.1
statement that delegates content to the DSAs (15) or sets an overridable default (6), x each DSA, read by
`sources/template_map.csv` and `sources/response_rules.csv` (`scripts/10_delegation_readings.py`).
The two-intervention defaults (no RAR, the RAR cap) also have one reading per illness-severity state.
Map, rules and responses approved by Kerstin 2026-10-25; all analyst readings. Statistical rows are for a
trial-design reviewer.


In [9]:
dr = pd.read_csv(BASE / "output" / "step5" / "delegation_readings.csv", dtype=str, keep_default_na=False)
ds = pd.read_csv(BASE / "output" / "step5" / "delegation_readings_by_state.csv", dtype=str, keep_default_na=False)
r4 = pd.DataFrame(run("r4_unanswered_delegations.rq"), columns=["dsa", "topic", "response", "secondary"])
exp = dr[(dr.kind == "delegation") & (dr.role == "paired")
         & (dr.response.isin(["no_answer", "condition_not_stated", "defers_back"]) | (dr.secondary_response == "defers_back"))]
exp_set = {("rcap:DomainDesign_" + SHORT[r.dsa], r.topic, r.response, r.secondary_response) for r in exp.itertuples()}
assert set(map(tuple, r4.values.tolist())) == exp_set and len(r4) == len(exp)
r4


,dsa,topic,response,secondary
0,rcap:DomainDesign_AB,discontinuation criteria,fills,defers_back
1,rcap:DomainDesign_AB,separate triggers,condition_not_stated,
2,rcap:DomainDesign_CS,discontinuation criteria,no_answer,defers_back
3,rcap:DomainDesign_CS,informative priors,condition_not_stated,
4,rcap:DomainDesign_CS,separate triggers,condition_not_stated,
5,rcap:DomainDesign_END,discontinuation criteria,fills,defers_back
6,rcap:DomainDesign_END,separate triggers,condition_not_stated,
7,rcap:DomainDesign_IAV,discontinuation criteria,fills,defers_back
8,rcap:DomainDesign_IAV,separate triggers,condition_not_stated,
9,rcap:DomainDesign_IG,discontinuation criteria,no_answer,defers_back


In [10]:
r5 = pd.DataFrame(run("r5_defaults.rq"), columns=["dsa", "topic", "response", "state", "state_response", "arms"])
pair = dr[dr.kind == "default"]
exp_pair = {("rcap:DomainDesign_" + SHORT[r.dsa], r.topic, r.response) for r in pair.itertuples()}
assert set(r5[["dsa", "topic", "response"]].itertuples(index=False, name=None)) == exp_pair
st = r5[r5.state != ""]
exp_state = {("rcap:DomainDesign_" + SHORT[r.dsa], r.topic, r.state, r.response, r.n_arms) for r in ds.itertuples()}
assert set(st[["dsa", "topic", "state", "state_response", "arms"]].itertuples(index=False, name=None)) == exp_state
assert len(st) == len(ds)
display(r5[r5.state == ""].pivot(index="topic", columns="dsa", values="response"))
st


dsa,rcap:DomainDesign_AB,rcap:DomainDesign_CS,rcap:DomainDesign_END,rcap:DomainDesign_IAV,rcap:DomainDesign_IG,rcap:DomainDesign_IIM,rcap:DomainDesign_MD,rcap:DomainDesign_MV,rcap:DomainDesign_TAB
topic,,,,,,,,,
balanced allocation at domain start,silent,silent,silent,silent,silent,silent,silent,silent,silent
new intervention at 1/K,silent,silent,silent,silent,silent,silent,silent,silent,silent
no control arm: no min/max,ambiguous,ambiguous,ambiguous,ambiguous,ambiguous,ambiguous,ambiguous,ambiguous,ambiguous
primary outcome applies,takes,takes,takes,takes,takes,takes,takes,takes,takes


,dsa,topic,response,state,state_response,arms
0,rcap:DomainDesign_AB,RAR cap for two interventions,silent,Severe,condition_not_met,4
1,rcap:DomainDesign_CS,RAR cap for two interventions,takes,Moderate,takes,2
2,rcap:DomainDesign_CS,RAR cap for two interventions,takes,Severe,condition_not_met,3
3,rcap:DomainDesign_END,RAR cap for two interventions,takes,Severe,takes,2
4,rcap:DomainDesign_IAV,RAR cap for two interventions,takes,Moderate,condition_not_met,4
5,rcap:DomainDesign_IAV,RAR cap for two interventions,takes,Severe,condition_not_met,4
6,rcap:DomainDesign_IG,RAR cap for two interventions,silent,Moderate,silent,2
7,rcap:DomainDesign_IG,RAR cap for two interventions,silent,Severe,silent,2
8,rcap:DomainDesign_IIM,RAR cap for two interventions,takes,Severe,condition_not_met,3
9,rcap:DomainDesign_MD,RAR cap for two interventions,silent,Severe,silent,2


## R2: what a DSA inherits before it states anything, against Tropical Antibiotic 1

- R2a: the components of Core 4.1 (V-current) and PANDA 1.0, per occurrence kind.
- R2b: the Core 4.1 overridable defaults (hold while the DSA is silent) and delegations (slots to fill).
- R2c: TAB 1 per kind: `points_at` = inherited components in the sections TAB's inheritance statements
  resolve to (part 1); `restates` = TAB components that Core 4.1 or PANDA also state; `states_own_*` = TAB
  components no Core 4.1 or PANDA text states, shared with another DSA or TAB only.
- R2d: TAB's answers to the slots and defaults (part 2), and its pointers to sections without components.
- R2e: the same counts for all DSAs in scope: components, restatements, inherited components reached and not
  reached by the DSA's inheritance statements, silent defaults.


In [11]:
r2a = pd.DataFrame(run("r2a_inherited_by_kind.rq"), columns=["kind", "n"])
r2a["n"] = r2a.n.astype(int)
up = occ[occ.doc.isin(["CORE41", "PANDA"])]
exp = up.drop_duplicates(["component", "kind"]).groupby("kind").size()
assert r2a.set_index("kind").n.to_dict() == exp.to_dict()
r2b = pd.DataFrame(run("r2b_slots_and_defaults.rq"), columns=["allocation_kind", "role", "topic", "conditional"])
exp = dr.assign(allocation_kind=dr.kind.map({"delegation": "delegation", "default": "overridable default"})) \
    .drop_duplicates(["allocation_kind", "role", "topic", "conditional"])
assert len(r2b) == len(exp) == 21
assert set(r2b.itertuples(index=False, name=None)) == \
    set(exp[["allocation_kind", "role", "topic", "conditional"]].itertuples(index=False, name=None))
print(up.component.nunique(), "inherited components,", r2a.n.sum(), "component-kind pairs")
r2b.groupby(["allocation_kind", "role"]).size().rename("Core 4.1 statements")


219 inherited components, 222 component-kind pairs


allocation_kind      role       
delegation           descriptive     1
                     paired         13
                     umbrella        1
overridable default  paired          6
Name: Core 4.1 statements, dtype: int64

In [12]:
r2c = pd.DataFrame(run("r2c_tab_by_kind.rq"), columns=["kind", "how", "n"])
r2c["n"] = r2c.n.astype(int)
up_c = set(up.component)
other_c = set(occ[occ.doc.isin(["CS_DSA", "AB_DSA", "IAV_DSA"])].component)
tab = occ[occ.doc == "TAB_DSA"].drop_duplicates(["component", "kind"])
tab = tab.assign(how=["restates" if c in up_c else "states_own_shared" if c in other_c else "states_own_TAB_only"
                      for c in tab.component])
ri = pd.read_csv(BASE / "output" / "step5" / "readings_index.csv", dtype=str, keep_default_na=False)
dep = set()
for r in ri[(ri.doc == "TAB_DSA") & (ri.resolution != "")].itertuples():
    for t in r.resolution.split(";"):
        layer, num = t.split(" ")
        docs = ["PANDA"] if layer == "PANDA" else (["CORE41", "CORE40"] if "4.0" in r.core_versions else ["CORE41"])
        dep |= set(occ[occ.doc.isin(docs) & (occ.section == num)].component)
pts = up[up.component.isin(dep)].drop_duplicates(["component", "kind"]).assign(how="points_at")
exp = pd.concat([tab, pts]).groupby(["kind", "how"]).size()
assert {(r.kind, r.how): r.n for r in r2c.itertuples()} == exp.to_dict()
table = r2c.pivot(index="kind", columns="how", values="n").join(r2a.set_index("kind").n.rename("inherited"), how="outer")
for col in ["points_at", "restates", "states_own_shared", "states_own_TAB_only"]:
    if col not in table:
        table[col] = 0
table = table[["inherited", "points_at", "restates", "states_own_shared", "states_own_TAB_only"]].fillna(0).astype(int)
display(table.sum().rename("total"))
table


inherited              222
points_at              109
restates                 0
states_own_shared       59
states_own_TAB_only     16
Name: total, dtype: int64

,inherited,points_at,restates,states_own_shared,states_own_TAB_only
kind,,,,,
allocation_interactions,0,0,0,2,0
allocation_nesting,0,0,0,0,1
allocation_rar,34,34,0,0,1
allocation_reveal,0,0,0,1,0
allocation_threshold,0,0,0,3,0
allocation_unit_of_analysis,0,0,0,2,0
applicability_age,0,0,0,1,0
applicability_state,0,0,0,1,0
category_acquisition,5,0,0,0,0


In [13]:
tabr = dr[(dr.dsa == "TAB_DSA") & (dr.role == "paired")]
display(pd.crosstab([tabr.kind, tabr.topic], tabr.response))
ri[(ri.doc == "TAB_DSA") & (ri.depends_on_components.astype(int) == 0)][["section", "resolution", "review_status"]]


response                                             ambiguous  \
kind       topic                                                 
default    RAR cap for two interventions                     0   
           balanced allocation at domain start               0   
           new intervention at 1/K                           0   
           no control arm: no min/max                        1   
           primary outcome applies                           0   
           two interventions: no RAR                         0   
delegation SAE clarification                                 0   
           additional variables                              0   
           blinding details                                  0   
           co-interventions                                  0   
           discontinuation criteria                          0   
           domain-specific sampling                          0   
           established safety profile and reporting          0   
           gamma and unit of analysis                        0   
           informative priors                                0   
           lambda per domain pair                            0   
           nesting                                           0   
           open-label or blinded                             0   
           separate triggers                                 0   

response                                             condition_not_stated  \
kind       topic                                                            
default    RAR cap for two interventions                                0   
           balanced allocation at domain start                          0   
           new intervention at 1/K                                      0   
           no control arm: no min/max                                   0   
           primary outcome applies                                      0   
           two interventions: no RAR                                    0   
delegation SAE clarification                                            0   
           additional variables                                         0   
           blinding details                                             0   
           co-interventions                                             0   
           discontinuation criteria                                     0   
           domain-specific sampling                                     0   
           established safety profile and reporting                     0   
           gamma and unit of analysis                                   0   
           informative priors                                           0   
           lambda per domain pair                                       0   
           nesting                                                      0   
           open-label or blinded                                        0   
           separate triggers                                            1   

response                                             fills  fills_none  \
kind       topic                                                         
default    RAR cap for two interventions                 0           0   
           balanced allocation at domain start           0           0   
           new intervention at 1/K                       0           0   
           no control arm: no min/max                    0           0   
           primary outcome applies                       0           0   
           two interventions: no RAR                     0           0   
delegation SAE clarification                             0           1   
           additional variables                          1           0   
           blinding details                              1           0   
           co-interventions                              1           0   
           discontinuation criteria                      1           0   
           domain-specific sampli

,section,resolution,review_status
34,2,,"version_scoped: names the current version, no ..."
40,8.5.2,CORE 8.12.2,analyst-reading; rule accepted 2026-10-26
41,9.3,CORE 8.6,analyst-reading; rule agreed 2026-10-24
43,10.7,CORE 7.8.6,analyst-reading; rule agreed 2026-10-24
44,11.1,CORE 8.11,analyst-reading; rule accepted 2026-10-25
45,11.2,CORE 8.12.3,analyst-reading; rule accepted 2026-10-25
46,12.1,CORE 2.5,analyst-reading; rule accepted 2026-10-25


In [14]:
r2e = pd.DataFrame(run("r2e_per_dsa.rq"), columns=["dsa", "measure", "n"])
r2e["n"] = r2e.n.astype(int)
r2e = r2e.pivot(index="dsa", columns="measure", values="n") \
    .reindex(columns=["components", "restates", "reached", "unreached", "silent_defaults"]).fillna(0).astype(int)
rows = {}
for d in DSAS:
    own = set(occ[occ.doc == d].component)
    dep = set()
    for r in ri[(ri.doc == d) & (ri.resolution != "")].itertuples():
        for t in r.resolution.split(";"):
            layer, num = t.split(" ")
            docs = ["PANDA"] if layer == "PANDA" else (["CORE41", "CORE40"] if "4.0" in r.core_versions else ["CORE41"])
            dep |= set(occ[occ.doc.isin(docs) & (occ.section == num)].component)
    silent = dr[(dr.dsa == d) & (dr.kind == "default") & (dr.role == "paired") & (dr.response == "silent")]
    rows["rcap:DomainDesign_" + SHORT[d]] = {"components": len(own), "restates": len(own & up_c), "reached": len(up_c & dep),
                                             "unreached": len(up_c - dep), "silent_defaults": len(silent)}
exp = pd.DataFrame(rows).T[["components", "restates", "reached", "unreached", "silent_defaults"]]
assert r2e[exp.columns].sort_index().equals(exp.sort_index().astype(int))
assert (r2e.reached + r2e.unreached == len(up_c)).all()
r2e[exp.columns]


measure,components,restates,reached,unreached,silent_defaults
dsa,,,,,
rcap:DomainDesign_AB,95,0,106,113,3
rcap:DomainDesign_CS,86,0,106,113,2
rcap:DomainDesign_END,76,0,106,113,2
rcap:DomainDesign_IAV,67,0,100,119,2
rcap:DomainDesign_IG,65,0,106,113,3
rcap:DomainDesign_IIM,96,0,106,113,2
rcap:DomainDesign_MD,52,0,106,113,3
rcap:DomainDesign_MV,64,0,106,113,4
rcap:DomainDesign_TAB,75,0,106,113,3


## Crosswalk: exact identity, decisions, upper bound

`exact`: identity by exact text only. `decided`: Kerstin's decisions applied (the modules above).
`all_pairs_merged`: every review pair merged, an upper bound only (many pairs differ in substance).


In [15]:
cw = pd.read_csv(BASE / "output" / "step3" / "crosswalk_review.csv", dtype=str, keep_default_na=False)
dec = pd.read_csv(BASE / "sources" / "crosswalk_decisions.csv", dtype=str, keep_default_na=False)
assert (dec.decision != "").all()
exact_docs = occ.groupby("component_exact").doc.apply(lambda v: frozenset(v))
parent = {}


def find(k):
    parent.setdefault(k, k)
    while parent[k] != k:
        parent[k] = parent[parent[k]]
        k = parent[k]
    return k


for r in cw.itertuples():
    parent[find(r.component_b)] = find(r.component_a)
merged_docs = {}
for c, d in exact_docs.items():
    merged_docs.setdefault(find(c), set()).update(d)
variants = {"exact": [set(d) for d in exact_docs.tolist()], "decided": [set(d) for d in docs_of.tolist()],
            "all_pairs_merged": list(merged_docs.values())}


def shared(sets, a, b):
    return sum(1 for d in sets if {a, b} <= d and not (d & set(NOT_DSA)))


rows = [{"question": "R1 all DSAs in scope", **{k: sum(1 for d in v if d == set(DSAS)) for k, v in variants.items()}}]
for i, a in enumerate(DSAS):
    for b in DSAS[i + 1:]:
        rows.append({"question": "R3 " + SHORT[a] + "-" + SHORT[b], **{k: shared(v, a, b) for k, v in variants.items()}})
table = pd.DataFrame(rows)
assert table.loc[0, "decided"] == len(r1)
table


,question,exact,decided,all_pairs_merged
0,R1 all DSAs in scope,1,4,6
1,R3 CS-AB,12,18,22
2,R3 CS-TAB,11,17,22
3,R3 CS-IAV,14,24,28
4,R3 CS-MD,8,11,14
5,R3 CS-IG,7,16,19
6,R3 CS-MV,6,12,13
7,R3 CS-END,10,16,22
8,R3 CS-IIM,16,24,27
9,R3 AB-TAB,52,58,63
